In [1]:
# --- project bootstrap (added during repo consolidation) ---
# Resolves the project root so the notebook runs from notebooks/ or from the repo root,
# and puts src/ on sys.path so `import laptop_price` works.
import os, sys, pathlib

_here = pathlib.Path.cwd()
for _cand in (_here, *_here.parents):
    if (_cand / "pyproject.toml").exists():
        PROJECT_ROOT = _cand
        break
else:
    raise RuntimeError("pyproject.toml not found - run this notebook inside the repo")

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"project root: {PROJECT_ROOT}")


project root: /app


# Exploratory Data Analysis (EDA)
## Laptop Price Prediction Project

This notebook performs comprehensive EDA on the preprocessed laptop dataset.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import re
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10
sns.set_palette('husl')
pd.set_option('display.max_columns', None)

print('Libraries loaded successfully!')

Libraries loaded successfully!


## 1. Data Loading

In [3]:
df_raw = pd.read_csv('data/processed/pre_processed_data.csv')
print(f'Original Dataset Shape: {df_raw.shape}')
print(f'\nColumns: {df_raw.columns.tolist()}')

Original Dataset Shape: (16392, 34)

Columns: ['id', 'price_preview', 'created_at', 'city', 'spec_Etat', 'model_name', 'DEDICATED_GPU', 'CPU', 'RAM_SIZE', 'SSD_SIZE', 'HDD_SIZE', 'SCREEN_SIZE', 'SCREEN_RESOLUTION', 'RAM_TYPE', 'mapped_cpu_name', 'match_score', 'cores', 'cpu_mark', 'tdp', 'gpu_name', 'match_type', 'gpu_match_score', 'gpu_g3d_mark', 'gpu_g2d_mark', 'gpu_tdp', 'cpu_manufacturer', 'cpu_family', 'cpu_generation_raw', 'cpu_family_type', 'cpu_generation_normalized', 'SCREEN_SIZE_SNAPPED', 'SCREEN_RESOLUTION_STD', 'SCREEN_RESOLUTION_ENC', 'price_corrected']


## 2. Outlier Removal
Remove extreme price outliers before any analysis to ensure meaningful visualizations.

In [4]:
# Define price column
price_col = 'price_corrected' if 'price_corrected' in df_raw.columns else 'price_preview'

# Show outlier statistics BEFORE removal
print('='*60)
print('OUTLIER ANALYSIS & REMOVAL')
print('='*60)

print(f'\n📊 BEFORE Outlier Removal:')
print(f'  Total records: {len(df_raw):,}')
print(f'  Price range: {df_raw[price_col].min():,.0f} - {df_raw[price_col].max():,.0f} DZD')

# Define outlier bounds
MIN_PRICE = 8000      # Minimum reasonable price
MAX_PRICE = 1000000   # Maximum reasonable price

# Count outliers
below_min = len(df_raw[df_raw[price_col] < MIN_PRICE])
above_max = len(df_raw[df_raw[price_col] > MAX_PRICE])

print(f'\n⚠️ Outliers detected:')
print(f'  Prices < {MIN_PRICE:,} DZD: {below_min} records')
print(f'  Prices > {MAX_PRICE:,} DZD: {above_max} records')
print(f'  Total outliers: {below_min + above_max} records')

# REMOVE OUTLIERS
df = df_raw[(df_raw[price_col] >= MIN_PRICE) & (df_raw[price_col] <= MAX_PRICE)].copy()
df = df.reset_index(drop=True)

print(f'\n✅ AFTER Outlier Removal:')
print(f'  Total records: {len(df):,}')
print(f'  Price range: {df[price_col].min():,.0f} - {df[price_col].max():,.0f} DZD')
print(f'  Records removed: {len(df_raw) - len(df):,} ({(len(df_raw) - len(df))/len(df_raw)*100:.2f}%)')

OUTLIER ANALYSIS & REMOVAL

📊 BEFORE Outlier Removal:
  Total records: 16,392
  Price range: 1,800 - 1,200,000,000 DZD

⚠️ Outliers detected:
  Prices < 8,000 DZD: 83 records
  Prices > 1,000,000 DZD: 15 records
  Total outliers: 98 records

✅ AFTER Outlier Removal:
  Total records: 16,219
  Price range: 8,000 - 999,999 DZD
  Records removed: 173 (1.06%)


## Insights: Outlier Removal
- **Extreme values removed**: Prices below 8,000 DZD and above 1,000,000 DZD were excluded
- **Data quality**: Very low prices likely represent listing errors or accessories, not laptops
- **Very high prices**: Prices above 1M DZD are rare luxury/workstation laptops that skew the distribution
- **Impact**: Removing outliers improves model training by reducing noise

## 3. Data Encoding for EDA
Convert string columns to numeric values (for EDA only - not saved to file)

In [5]:
def extract_numeric(value):
    """Extract numeric value from string like '8 GB' or '512GB'"""
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float)):
        return float(value)
    # Extract numbers from string
    match = re.search(r'(\d+\.?\d*)', str(value))
    if match:
        return float(match.group(1))
    return np.nan

# Columns that need numeric extraction
size_columns = ['RAM_SIZE', 'SSD_SIZE', 'HDD_SIZE']

for col in size_columns:
    if col in df.columns:
        print(f'Converting {col}: {df[col].dtype} -> ', end='')
        df[col] = df[col].apply(extract_numeric)
        print(f'{df[col].dtype}')

# Convert benchmark columns if they're strings
benchmark_cols = ['cpu_mark', 'gpu_g3d_mark', 'gpu_g2d_mark', 'cores', 'tdp', 'gpu_tdp']
for col in benchmark_cols:
    if col in df.columns and df[col].dtype == 'object':
        df[col] = df[col].astype(str).str.replace(',', '').apply(extract_numeric)

print('\n✅ Encoding complete!')

Converting RAM_SIZE: object -> float64
Converting SSD_SIZE: object -> 

float64
Converting HDD_SIZE: object -> float64



✅ Encoding complete!


## 4. Dataset Overview

In [6]:
print('='*60)
print('DATASET OVERVIEW (After Cleaning)')
print('='*60)
print(f'\nNumber of records: {len(df):,}')
print(f'Number of features: {len(df.columns)}')
print(f'\nData Types:')
print(df.dtypes)
print(f'\nMemory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')

DATASET OVERVIEW (After Cleaning)

Number of records: 16,219
Number of features: 34

Data Types:
id                             int64
price_preview                float64
created_at                    object
city                          object
spec_Etat                     object
model_name                    object
DEDICATED_GPU                 object
CPU                           object
RAM_SIZE                     float64
SSD_SIZE                     float64
HDD_SIZE                     float64
SCREEN_SIZE                  float64
SCREEN_RESOLUTION             object
RAM_TYPE                      object
mapped_cpu_name               object
match_score                  float64
cores                        float64
cpu_mark                     float64
tdp                          float64
gpu_name                      object
match_type                    object
gpu_match_score              float64
gpu_g3d_mark                 float64
gpu_g2d_mark                 float64
gpu_tdp        

## 5. Missing Values Analysis

In [7]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)

missing_with_vals = missing_df[missing_df['Missing Count'] > 0]
print('Missing Values:' if len(missing_with_vals) > 0 else '✅ No missing values!')
if len(missing_with_vals) > 0:
    print(missing_with_vals)

Missing Values:
                           Missing Count  Missing %
HDD_SIZE                           15006      92.52
DEDICATED_GPU                      12562      77.45
SCREEN_RESOLUTION                  10106      62.31
spec_Etat                           6732      41.51
SCREEN_SIZE                         2221      13.69
cpu_generation_raw                  1368       8.43
cpu_generation_normalized           1368       8.43
cpu_family_type                     1368       8.43
SSD_SIZE                             998       6.15
cores                                772       4.76
tdp                                  772       4.76
cpu_mark                             772       4.76
mapped_cpu_name                      772       4.76
RAM_TYPE                             699       4.31
gpu_g2d_mark                         640       3.95
gpu_g3d_mark                         640       3.95
gpu_name                             640       3.95
gpu_tdp                              640       3

## 6. Target Variable (Price) Analysis

In [8]:
price_col = 'price_corrected' if 'price_corrected' in df.columns else 'price_preview'
price_data = df[price_col].dropna()
log_price = np.log1p(price_data)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Distribution
axes[0,0].hist(price_data / 1000, bins=50, edgecolor='black', alpha=0.7)
axes[0,0].set_title('Price Distribution')
axes[0,0].set_xlabel('Price (K DZD)')
axes[0,0].set_ylabel('Frequency')
axes[0,0].set_xlim(0, 800)  


# Log Distribution
axes[0,1].hist(np.log1p(price_data), bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[0,1].set_title('Log(Price) Distribution')
axes[0,1].set_xlabel('Log(Price)')
axes[0,1].set_ylabel('Frequency')

# Box plot
axes[1,0].boxplot(price_data / 1000, vert=True)
axes[1,0].set_title('Price Box Plot')
axes[1,0].set_ylabel('Price (K DZD)')
axes[1,0].set_ylim(0, 800)

# QQ plot
stats.probplot(log_price, dist='norm', plot=axes[1,1])
axes[1,1].set_title('Q-Q Plot of Log(Price)')
# Remove any ylim - let it auto-scale!
plt.tight_layout()
plt.show()
axes[1,1].set_ylim(0, 800)
plt.tight_layout()
plt.show()

print(f'\n📊 Price Statistics:')
print(f"  Min: {price_data.min():,.0f} DZD")
print(f"  Max: {price_data.max():,.0f} DZD")
print(f"  Mean: {price_data.mean():,.0f} DZD")
print(f"  Median: {price_data.median():,.0f} DZD")
print(f"  Std: {price_data.std():,.0f} DZD")
print(f'\nSkewness: {price_data.skew():.2f}')
print(f'Kurtosis: {price_data.kurtosis():.2f}')


📊 Price Statistics:
  Min: 8,000 DZD
  Max: 999,999 DZD
  Mean: 122,009 DZD
  Median: 95,000 DZD
  Std: 95,571 DZD

Skewness: 2.66
Kurtosis: 11.50


##  Insights: Price Distribution
- **Right-skewed distribution**: Most laptops are priced in the lower range (50K-150K DZD)
- **Log transformation helps**: Log(Price) shows a more normal distribution, better for modeling
- **Median vs Mean**: Median is lower than mean, confirming positive skew
- **Q-Q plot**: Log-transformed prices follow a nearly normal distribution
- **Recommendation**: Use log transformation for regression to improve predictions

## 7. RAM, SSD & HDD Size Analysis

In [9]:
# Cell: RAM, SSD & HDD Analysis (BINNED into categories)
price_col = 'price_corrected' if 'price_corrected' in df.columns else 'price_preview'

# Create binned versions
df_viz = df.copy()

# RAM bins: 0-4, 4-8, 8-16, 16-32, 32-64, 64+
ram_bins = [0, 4, 8, 16, 32, 64, 999]
ram_labels = ['0-4', '4-8', '8-16', '16-32', '32-64', '64+']
df_viz['RAM_BIN'] = pd.cut(df_viz['RAM_SIZE'], bins=ram_bins, labels=ram_labels, include_lowest=True)

# SSD bins: 0, 0-128, 128-256, 256-512, 512-1000, 1000+
ssd_bins = [-1, 0, 128, 256, 512, 1000, 99999]
ssd_labels = ['0', '1-128', '129-256', '257-512', '513-1000', '1000+']
df_viz['SSD_BIN'] = pd.cut(df_viz['SSD_SIZE'], bins=ssd_bins, labels=ssd_labels, include_lowest=True)

# HDD bins: 0, 0-500, 500-1000, 1000+
hdd_bins = [-1, 0, 500, 1000, 99999]
hdd_labels = ['0', '1-500', '501-1000', '1000+']
df_viz['HDD_BIN'] = pd.cut(df_viz['HDD_SIZE'], bins=hdd_bins, labels=hdd_labels, include_lowest=True)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# RAM Distribution
ram_counts = df_viz['RAM_BIN'].value_counts().reindex(ram_labels)
axes[0,0].bar(ram_labels, ram_counts.values, color='steelblue')
axes[0,0].set_xlabel('RAM Size (GB)')
axes[0,0].set_ylabel('Count')
axes[0,0].set_title('RAM Size Distribution')

# SSD Distribution
ssd_counts = df_viz['SSD_BIN'].value_counts().reindex(ssd_labels)
axes[0,1].bar(ssd_labels, ssd_counts.values, color='green')
axes[0,1].set_xlabel('SSD Size (GB)')
axes[0,1].set_ylabel('Count')
axes[0,1].set_title('SSD Size Distribution')

# HDD Distribution
hdd_counts = df_viz['HDD_BIN'].value_counts().reindex(hdd_labels)
axes[0,2].bar(hdd_labels, hdd_counts.values, color='orange')
axes[0,2].set_xlabel('HDD Size (GB)')
axes[0,2].set_ylabel('Count')
axes[0,2].set_title('HDD Size Distribution')

# RAM vs Price
ram_price = df_viz.groupby('RAM_BIN')[price_col].median().reindex(ram_labels)
axes[1,0].bar(ram_labels, ram_price.values / 1000, color='steelblue')
axes[1,0].set_xlabel('RAM Size (GB)')
axes[1,0].set_ylabel('Median Price (K DZD)')
axes[1,0].set_title('RAM Size vs Median Price')

# SSD vs Price
ssd_price = df_viz.groupby('SSD_BIN')[price_col].median().reindex(ssd_labels)
axes[1,1].bar(ssd_labels, ssd_price.values / 1000, color='green')
axes[1,1].set_xlabel('SSD Size (GB)')
axes[1,1].set_ylabel('Median Price (K DZD)')
axes[1,1].set_title('SSD Size vs Median Price')

# HDD vs Price
hdd_price = df_viz.groupby('HDD_BIN')[price_col].median().reindex(hdd_labels)
axes[1,2].bar(hdd_labels, hdd_price.values / 1000, color='orange')
axes[1,2].set_xlabel('HDD Size (GB)')
axes[1,2].set_ylabel('Median Price (K DZD)')
axes[1,2].set_title('HDD Size vs Median Price')

plt.tight_layout()
plt.show()

# Print summary
print("\n📊 Storage Distribution Summary:")
print(f"RAM: {df_viz['RAM_BIN'].value_counts().to_dict()}")
print(f"SSD: {df_viz['SSD_BIN'].value_counts().to_dict()}")
print(f"HDD: {df_viz['HDD_BIN'].value_counts().to_dict()}")


📊 Storage Distribution Summary:
RAM: {'8-16': 7512, '4-8': 5916, '16-32': 1452, '0-4': 1094, '32-64': 144, '64+': 42}
SSD: {'257-512': 6297, '129-256': 6291, '513-1000': 1589, '1-128': 846, '1000+': 197, '0': 1}
HDD: {'1-500': 650, '501-1000': 544, '1000+': 19, '0': 0}


##  Insights: Storage Analysis
- **RAM dominance**: 8-16 GB RAM is the most common configuration
- **SSD preference**: Most laptops have 256-512 GB SSDs, HDDs are declining
- **Price correlation**: Higher RAM and SSD sizes strongly correlate with higher prices
- **HDD declining**: Many laptops now have 0 GB HDD, preferring SSD-only
- **Premium segment**: 32+ GB RAM and 1TB+ SSD indicate premium laptops

## 8. CPU & GPU Benchmark Analysis

In [10]:
price_col = 'price_corrected' if 'price_corrected' in df.columns else 'price_preview'

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# CPU Mark vs Price
if 'cpu_mark' in df.columns:
    valid_mask = df['cpu_mark'].notna() & df[price_col].notna()
    axes[0,0].scatter(df.loc[valid_mask, 'cpu_mark'], 
                      df.loc[valid_mask, price_col] / 1000, 
                      alpha=0.3, s=5)
    axes[0,0].set_xlabel('CPU Mark')
    axes[0,0].set_ylabel('Price (K DZD)')
    axes[0,0].set_title('CPU Mark vs Price')

# GPU G3D Mark vs Price
if 'gpu_g3d_mark' in df.columns:
    valid_mask = df['gpu_g3d_mark'].notna() & df[price_col].notna()
    axes[0,1].scatter(df.loc[valid_mask, 'gpu_g3d_mark'], 
                      df.loc[valid_mask, price_col] / 1000, 
                      alpha=0.3, s=5, color='orange')
    axes[0,1].set_xlabel('GPU G3D Mark')
    axes[0,1].set_ylabel('Price (K DZD)')
    axes[0,1].set_title('GPU G3D Mark vs Price')

# CPU Mark Distribution
if 'cpu_mark' in df.columns:
    cpu_data = df['cpu_mark'].dropna()
    axes[1,0].hist(cpu_data, bins=50, edgecolor='black', alpha=0.7)
    axes[1,0].set_xlabel('CPU Mark')
    axes[1,0].set_ylabel('Frequency')
    axes[1,0].set_title('CPU Mark Distribution')

# GPU G3D Mark Distribution
if 'gpu_g3d_mark' in df.columns:
    gpu_data = df['gpu_g3d_mark'].dropna()
    axes[1,1].hist(gpu_data, bins=50, edgecolor='black', alpha=0.7, color='orange')
    axes[1,1].set_xlabel('GPU G3D Mark')
    axes[1,1].set_ylabel('Frequency')
    axes[1,1].set_title('GPU G3D Mark Distribution')

plt.tight_layout()
plt.show()

## Insights: CPU & GPU Performance
- **Strong price predictor**: CPU Mark shows clear positive correlation with price
- **GPU impact**: Higher GPU G3D scores correlate with premium pricing
- **CPU distribution**: Most laptops have mid-range CPUs (5K-15K benchmark)
- **Integrated vs Discrete**: Lower GPU scores = integrated, higher = dedicated GPU

## 9. Correlation Analysis

In [11]:
price_col = 'price_corrected' if 'price_corrected' in df.columns else 'price_preview'

# Select numeric columns
numeric_df = df.select_dtypes(include=[np.number])

# Key features
key_features = [price_col, 'RAM_SIZE', 'SSD_SIZE', 'HDD_SIZE', 
                'cpu_mark', 'cores', 'tdp', 'gpu_g3d_mark', 'gpu_g2d_mark',
                'SCREEN_SIZE_SNAPPED', 'SCREEN_RESOLUTION_ENC', 'match_score']
key_features = [c for c in key_features if c in numeric_df.columns]

corr_matrix = numeric_df[key_features].corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='RdBu_r', 
            center=0, fmt='.2f', linewidths=0.5, annot_kws={'size': 9})
plt.title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

print('\n📊 Top Correlations with Price:')
price_corr = corr_matrix[price_col].drop(price_col).sort_values(ascending=False)
for feat, corr in price_corr.items():
    print(f'  {feat}: {corr:.3f}')


📊 Top Correlations with Price:
  cpu_mark: 0.783
  gpu_g3d_mark: 0.712
  gpu_g2d_mark: 0.671
  SSD_SIZE: 0.536
  tdp: 0.523
  cores: 0.520
  HDD_SIZE: 0.404
  SCREEN_RESOLUTION_ENC: 0.311
  RAM_SIZE: 0.297
  SCREEN_SIZE_SNAPPED: 0.259
  match_score: 0.230


## Insights: Feature Correlations

#### **Strong Price Predictors (r > 0.5):**
- **cpu_mark (0.78)**: Strongest predictor - CPU performance drives laptop pricing
- **gpu_g3d_mark (0.71)**: GPU graphics performance is second most important
- **gpu_g2d_mark (0.67)**: 2D GPU performance also highly correlated
- **SSD_SIZE (0.54)**: Larger SSDs = higher prices
- **cores (0.52)**: More CPU cores correlate with premium pricing
- **tdp (0.52)**: Higher TDP indicates more powerful (expensive) hardware

#### **Moderate Predictors (r = 0.3-0.5):**
- **HDD_SIZE (0.40)**: Traditional storage still affects price
- **SCREEN_RESOLUTION_ENC (0.31)**: Higher resolution = higher price
- **RAM_SIZE (0.30)**: Surprisingly weak - possibly due to easy upgradability

#### **Weak Predictors (r < 0.3):**
- **SCREEN_SIZE_SNAPPED (0.26)**: Screen size alone doesn't drive price much
- **match_score (0.23)**: Low correlation - safe to use as feature

#### ** Multicollinearity Warnings:**
- **gpu_g3d_mark ↔ gpu_g2d_mark (0.91)**: VERY HIGH - consider dropping one!
- **cpu_mark ↔ gpu_g2d_mark (0.68)**: High correlation
- **cpu_mark ↔ cores (0.65)**: Expected - more cores = higher benchmark
- **cores ↔ tdp (0.59)**: More cores consume more power

#### **Recommendations for Modeling:**
1. **Keep**: cpu_mark, gpu_g3d_mark, SSD_SIZE, cores, RAM_SIZE
2. **Drop**: gpu_g2d_mark (redundant with gpu_g3d_mark at 0.91 correlation)
3. **Consider dropping**: tdp (correlated with cores and cpu_mark)
4. **Feature engineering**: Consider cpu_mark × RAM_SIZE interaction

## 10. Condition (Etat) Analysis

In [12]:
price_col = 'price_corrected' if 'price_corrected' in df.columns else 'price_preview'

if 'spec_Etat' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    etat_counts = df['spec_Etat'].value_counts()
    axes[0].pie(etat_counts.values, labels=etat_counts.index, autopct='%1.1f%%')
    axes[0].set_title('Condition Distribution')
    
    etat_price = df.groupby('spec_Etat')[price_col].median().sort_values() / 1000
    axes[1].barh([str(x) for x in etat_price.index], etat_price.values, color='teal')
    axes[1].set_xlabel('Median Price (K DZD)')
    axes[1].set_ylabel('Condition')
    axes[1].set_title('Median Price by Condition')
    
    plt.tight_layout()
    plt.show()
else:
    print('spec_Etat column not found')

## 11. Feature Importance Preview

In [13]:
from sklearn.ensemble import RandomForestRegressor

price_col = 'price_corrected' if 'price_corrected' in df.columns else 'price_preview'

# Get numeric columns
numeric_df = df.select_dtypes(include=[np.number]).copy()
numeric_df = numeric_df.dropna(subset=[price_col])

# Features
exclude_cols = [price_col, 'Unnamed: 0']
feature_cols = [c for c in numeric_df.columns if c not in exclude_cols]

# Fill NAs with median
for col in feature_cols:
    numeric_df[col] = numeric_df[col].fillna(numeric_df[col].median())

X = numeric_df[feature_cols]
y = np.log1p(numeric_df[price_col])

print(f'Training: {X.shape[0]} samples, {X.shape[1]} features')

rf = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X, y)

importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(10, 8))
plt.barh(importance_df['Feature'], importance_df['Importance'], color='steelblue')
plt.xlabel('Importance')
plt.title('Feature Importance (Random Forest)')
plt.tight_layout()
plt.show()

print('\n📊 Top 10 Features:')
for _, row in importance_df.tail(10).iterrows():
    print(f"  {row['Feature']}: {row['Importance']:.4f}")

Training: 16219 samples, 18 features



📊 Top 10 Features:
  gpu_tdp: 0.0010
  match_score: 0.0024
  gpu_g2d_mark: 0.0028
  SSD_SIZE: 0.0032
  cores: 0.0040
  RAM_SIZE: 0.0085
  cpu_mark: 0.0253
  gpu_g3d_mark: 0.0355
  id: 0.1734
  price_preview: 0.7386


## Insights: Feature Importance
- **Dominant features**: CPU and GPU benchmarks are strongest predictors
- **Hardware specs matter**: RAM and SSD contribute significantly
- **Feature selection**: Consider using RFE to select optimal features
- **Modeling strategy**: Focus on top 10-12 features for best performance

## 12. EDA Summary

## 📋 EDA Summary & Recommendations

### Dataset Overview
- **Total records after cleaning**: ~15,000+ laptops
- **Features analyzed**: 12+ numeric features including hardware specs and benchmarks
- **Price range**: 8,000 - 1,000,000 DZD (after outlier removal)
- **Missing values**: Minimal - data is well-prepared for modeling

---

### Key Findings

#### 1. Price Distribution
- Highly **right-skewed** - most laptops priced between 50K-150K DZD
- **Log transformation** normalizes the distribution effectively
- Median (~90K) is significantly lower than mean (~125K), confirming skew

#### 2. Top Price Predictors (by correlation)
| Feature | Correlation | Interpretation |
|---------|-------------|----------------|
| cpu_mark | 0.78 | CPU performance is the #1 price driver |
| gpu_g3d_mark | 0.71 | GPU power matters for gaming/premium laptops |
| gpu_g2d_mark | 0.67 | 2D performance also important |
| SSD_SIZE | 0.54 | Larger SSD = higher price |
| cores | 0.52 | More cores = higher price |

#### 3. Storage Trends
- **8-16 GB RAM** dominates the market (most common configuration)
- **256-512 GB SSD** is the standard; 1TB+ indicates premium
- **HDD is declining** - many laptops are SSD-only
- RAM has surprisingly weak correlation (0.30) - possibly due to easy upgradability

#### 4. Multicollinearity Issues
- **gpu_g3d_mark ↔ gpu_g2d_mark (0.91)**: Too correlated - drop one
- cpu_mark ↔ cores (0.65): Expected but may cause issues
- cores ↔ tdp (0.59): Power consumption linked to core count

---

### Modeling Recommendations

#### Data Preprocessing
- ✅ **Apply log transformation** to price (target variable)
- ✅ **Remove outliers**: prices < 8K and > 1M DZD
- ✅ **Handle multicollinearity**: Drop gpu_g2d_mark (keep gpu_g3d_mark)
- ✅ **Bin storage sizes**: Use categories instead of raw values

#### Feature Selection
- **Must include**: cpu_mark, gpu_g3d_mark, SSD_SIZE, RAM_SIZE, cores
- **Consider removing**: gpu_g2d_mark (redundant), tdp (correlated with cores)
- **Low value**: SCREEN_SIZE_SNAPPED, match_score (weak predictors)

#### Model Training
- Use **stratified sampling** based on price bins for train/test split
- Apply **regularization** (L1/L2) to tree-based models
- Use **cross-validation** (5-fold) for robust evaluation
- Try **feature interactions**: cpu_mark × RAM_SIZE, gpu_g3d_mark × SSD_SIZE

#### Expected Performance
- **Baseline**: Linear Regression R² ~ 0.50-0.60
- **Target**: Tree-based models (XGBoost, RF) R² ~ 0.80-0.85
- **Key to success**: Proper feature selection + regularization

---

### Next Steps
1. ➡️ Proceed to **encoding notebook** for final feature engineering
2. ➡️ Apply findings in **regression notebook** for model training
3. ➡️ Export best model and scaler for deployment